드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/part3data"

print(os.listdir(BASE_DIR))

# 1.데이터 구조 확인

## 1.1.폴더 구조 확인

In [ ]:
import os

def print_tree(path, max_depth=2):
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, '').count(os.sep)

        if depth > max_depth:
            continue

        indent = ' ' * 4 * depth
        print(f'{indent}{os.path.basename(root)}/')

        subindent = ' ' * 4 * (depth + 1)

        for f in files[:10]:
            print(f'{subindent}{f}')

print_tree(BASE_DIR)

**결과**

part3data /

    - data_list.csv

    - data_list.gsheet

    - files /

        남서울대학교_[혁신-국고]남서울대학교 스마트 정보시스템 활성화(학사.hwp

        (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp
        
        국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp
        
        한국해양조사협회_2024년 항해용 간행물 품질관리 업무보조 시스템 구축.hwp
        
        2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
        
        한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp
        
        한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp
        
        대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp
        
        한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp
        
        사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp

##1.2.데이터 구조


---

## 컬럼 구성 및 의미

| 컬럼명       | 설명                        | 활용                     |
| --------- | ------------------------- | ------------------------- |
| 공고 번호     | 조달청 또는 발주기관에서 부여한 고유 공고번호 | 사업 식별, 중복 공고 확인           |
| 공고 차수     | 동일 공고의 재공고 횟수             | 재공고 분석, 사업 반복 여부 확인       |
| 사업명       | 사업의 공식 명칭                 | 사업 유형 분류, 검색, 유사 사업 탐색    |
| 사업 금액     | 사업 예산 또는 계약 예정 금액         | 사업 규모 분석                  |
| 발주 기관     | 사업을 발주한 기관명               | 기관별 사업 분석                 |
| 공개 일자     | 공고가 게시된 일시                | 시계열 분석                    |
| 입찰 참여 시작일 | 입찰 접수 시작일                 | 사업 일정 분석                  |
| 입찰 참여 마감일 | 입찰 접수 종료일                 | 사업 일정 분석                  |
| 사업 요약     | 사업 목적 및 주요 내용 요약          | 검색, 질의응답, 요약 생성           |
| 파일형식      | 원본 문서 형식(pdf, hwp 등)      | 문서 처리 전략 수립               |
| 파일명       | 저장된 원본 문서 파일명             | 문서 매핑 및 추적                |
| 텍스트       | 원본 문서에서 추출한 일부 텍스트        | Retrieval 전 필터링, 문서 요약 정보 생성 |

---

## 주요 메타데이터

RAG 시스템 구축 및 분석에 활용 가능한 핵심 메타데이터

### 1. 발주기관

* 사업을 발주한 기관 정보
* 기관별 사업 특성 분석 가능
* 검색 필터링에 활용 가능

### 2. 사업명

* 사업 분야 및 목적 파악 가능
* 구축, 고도화, 운영, 유지보수 등 사업 유형 분류 가능
* 유사 사업 탐색 가능

### 3. 공고번호

* 사업의 고유 식별자
* 중복 데이터 검증 가능

### 4. 공고차수

* 재공고 여부 판단 가능
* 반복 공고 사업 분석 가능

### 5. 사업금액

* 사업 규모 분석 가능
* 예산 구간별 사업 분류 가능

### 6. 공개일자 및 입찰일정

* 사업 공고 시기 분석

### 7. 사업요약

* 사업 목적을 빠르게 파악 가능
* 검색 정확도 향상에 활용 가능

### 8. **원본문서 텍스트(중요)**

* `Retrieval 전 필터링` : 임베딩 전에 질문과 같은 키워드가 있는 문서를 후보군으로 둘 수 있음(정확도, 검색 속도 증가 )
* `BM25 + Vector Hybrid Search` : 메타데이터 텍스트를 BM25 인덱스에 같이 넣으면 같은 고유명사는 BM25가 강함
* `문서 요약 정보 생성`
* `사업명 정규화 생성 가능` : 사업명의 불용어들을 제거해 비슷한 사업의 경우 사업명을 정규화 하여 질문 시 유사 사업 관련 자료를 빠르게 검색하고 조회할 수 있음
* `Self-Query Retriever 사용 가능` : 사용자의 질문을 llm이 조건식으로 바꿔주면 컬럼들로 필터링이 가능해짐
* `Reranking Feature로 사용` : top 문서 뽑은것과 메타데이터 텍스트와 질문의 유사도 계산 후 가중 평균 후 사용
---

## 데이터 특징

* 메타데이터와 원본 문서가 연결된 구조
* PDF 및 HWP 문서 혼합 구성
* 공공기관 정보화 사업 제안요청서(RFP) 중심 데이터
* 사업 개요, 구축 범위, 요구사항, 평가 기준 등의 정보를 포함
* 검색 기반 QA(RAG) 구축에 적합한 데이터 구조를 가짐


In [ ]:
import pandas as pd

csv_path = f"{BASE_DIR}/data_list.csv"

df = pd.read_csv(csv_path)

print("행, 열:", df.shape)
print("\n컬럼 목록")
print(df.columns.tolist())

df.head()

**결과**

행, 열: (100, 12)

컬럼 목록
['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트']


## 1.4.파일명 확인

In [ ]:
from pathlib import Path

files_dir = Path(f"{BASE_DIR}/files")

for file in sorted(files_dir.iterdir())[:]:
    print(file.name)

# 결과

(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp

(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp

(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp

(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp

2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp

BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp

KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp

경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp

경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp

경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp

경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp

경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp

고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf

고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp

광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp

광주과학기술원_학사시스템 기능개선 사업.hwp

국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp

국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp

국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp

국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp
국민연금공단_2024년 이러닝시스템 운영 용역.hwp

국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp

국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp

국방과학연구소_대용량 자료전송시스템 고도화.hwp

그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf

나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp

남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp

대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp

대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp

대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp

대한장애인체육회_2025년 전국장애인체육대회 전산 및 시스템, 홈페이지 .hwp

대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .hwp

문화체육관광부 국립민속박물관_2024년 국립민속박물관 민속아카이브 자.hwp

부산관광공사_경영정보시스템 기능개선.hwp

사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp

사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp

서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp

서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp

서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf

서울특별시 여성가족재단_(재공고, 협상) 서울 디지털성범죄 안심지원센.hwp

서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf

서울특별시교육청_서울특별시교육청 지능정보화전략계획(ISP) 수립(2차) .hwp

세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp

수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp

수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp

울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp

을지대학교_을지대학교 비교과시스템 개발.hwp

인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp

인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp

인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp

인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp

재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp

재단법인 광주연구원_광주정책연구아카이브(GPA) 시스템 개발.hwp

재단법인 한국장애인문화예술원_2024년 장애인문화예술정보시스템 이음.hwp

재단법인경기도일자리재단_2025년 통합접수시스템 운영.hwp

재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp

재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp

전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp

전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp

조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp

중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp

축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp

축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp

케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp

파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp

한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp

한국건강가정진흥원_2025년 아이돌봄인력 인적성 검사 정보시스템 운영.hwp

한국교육과정평가원_국가교육과정정보센터(NCIC) 시스템 운영 및 개선.hwp

한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계시스템 .hwp

한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp

한국로봇산업진흥원_한국로봇산업진흥원 사업관리시스템 온라인평가 .hwp

한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp

한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp

한국보육진흥원_연차별 자율 품질관리 시스템 기능개선.hwp

한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고.hwp

한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp

한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp

한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp

한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp

한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp

한국수자원공사_건설통합시스템(CMS) 고도화.hwp

한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp

한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp

한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp

한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp

한국어촌어항공단_한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도.hwp

한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp

한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp

한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp

한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp

한국전기안전공사_전기안전 관제시스템 보안 모듈 개발 용역.hwp

한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp

한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp

한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp

한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp

한국한의학연구원_통합정보시스템 고도화 용역.hwp

한국해양조사협회_2024년 항해용 간행물 품질관리 업무보조 시스템 구축.hwp

한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp


# 2.EDA(메타 데이터)

## 2.1.메타데이터의 텍스트 길이 분포 분석

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df["문서길이"] = df["텍스트"].astype(str).str.len()

print(df["문서길이"].describe())

plt.figure(figsize=(10,5))
plt.hist(df["문서길이"], bins=30)
plt.title("Document Length Distribution")
plt.xlabel("Character Length")
plt.ylabel("Document Count")
plt.show()

원본 문서가 아닌 메타데이터의 요약 데이터 형태

**결과**

count      100.000000

mean      3843.530000

std       3692.593749

min         89.000000

25%       1198.000000

50%       2583.000000

75%       5842.000000

max      18335.000000


## 2.2.중복문서 탐지
- 파일명과 내용은 중복이 없으나 pk가 같은 문서가있음
-> 중복이 아니라 Na 중복탐지

> **중복 파일 없음**

재철님이 공유주신 파일 중복은 전처리할때 포함시키겠습니다!


In [ ]:
# primary key로 중복확인
duplicate_text = df["공고 번호"].duplicated().sum()

print(f"중복 문서 수: {duplicate_text}")
print(f"중복 비율: {duplicate_text/len(df)*100:.2f}%")

**결과**

중복 문서 수: 17

중복 비율: 17.00%

In [ ]:
# 파일명 중복확인
duplicate_text = df["파일명"].duplicated().sum()

print(f"중복 문서 수: {duplicate_text}")
print(f"중복 비율: {duplicate_text/len(df)*100:.2f}%")

**결과**

중복 문서 수: 0

중복 비율: 0.00%

In [ ]:
# 내용 중복확인
duplicate_text = df["텍스트"].duplicated().sum()

print(f"중복 문서 수: {duplicate_text}")
print(f"중복 비율: {duplicate_text/len(df)*100:.2f}%")

**결과**

중복 문서 수: 0

중복 비율: 0.00%


In [ ]:
duplicates = df[df["공고 번호"].duplicated(keep=False)]

duplicates.sort_values("공고 번호")

## 2.3.목차 존재여부
- 텍스트 컬럼에 목차가 87% 있어서 구조기반 청크에 사용할 수 있음
- 원문 내용과는 관계 없음


In [ ]:
df["목차존재"] = df["텍스트"].str.contains(
    r"목\s*차",
    regex=True,
    na=False
)

print(df["목차존재"].value_counts())

**결과**

목차존재

True     87

False    13


## 2.4.주요 헤더 패턴

In [ ]:
headers = [
    "사업개요",
    "추진배경",
    "구축범위",
    "사업범위",
    "제안요청",
    "요구사항",
    "평가기준",
    "제안서",
    "유지보수",
    "사업수행"
]

header_count = {}

for h in headers:
    header_count[h] = (
        df["텍스트"]
        .str.contains(h, na=False)
        .sum()
    )

header_df = (
    pd.DataFrame(
        header_count.items(),
        columns=["Header","Count"]
    )
    .sort_values("Count", ascending=False)
)

display(header_df)

**결과**
||Header|	Count|
|---|---|---|
|4	|제안요청	84|
|7	|제안서	81|
|5	|요구사항|	75|
|0	|사업개요|	66|
|1	|추진배경|	49|
|3	|사업범위|	44|
|9	|사업수행|	32|
|8	|유지보수|	27|
|6	|평가기준|	26|
|2	|구축범위|	2|

## 3.5.사업 유형 분석

In [ ]:
project_types = {
    "구축": "구축",
    "고도화": "고도화",
    "운영": "운영",
    "유지보수": "유지보수",
    "개선": "개선",
    "개발": "개발",
    "재구축": "재구축",
    "통합": "통합"
}

result = {}

for key in project_types:
    result[key] = df["사업명"].str.contains(
        key,
        na=False
    ).sum()

type_df = pd.DataFrame(
    result.items(),
    columns=["사업유형", "건수"]
)

type_df = type_df.sort_values(
    "건수",
    ascending=False
)

display(type_df)

**결과**
||사업유형	|건수|
|---|---|---|
|0|	구축	|42|
|7|	통합	|17|
|1|	고도화|	16|
|4|	개선	|15|
|2|	운영	|13|
|5|	개발|	8|
|6|	재구축	|2|
|3|	유지보수|	0|


## 3.6.사업명 유사도

In [ ]:
from collections import Counter
import re

all_titles = " ".join(df["사업명"].astype(str))

words = re.findall(r"[가-힣A-Za-z]+", all_titles)

counter = Counter(words)

top_words = pd.DataFrame(
    counter.most_common(30),
    columns=["단어","빈도"]
)

display(top_words)

||단어	|빈도|
|---|---|---|
|0|	용역	|37
|1|	구축	|31
|2|	시스템|	23
|3|	및	|19
|4|	고도화|	16
|5|	년	|16
|6|	사업|	14
|7|	기능개선|	11
|8|	긴급|	7
|9|	개발|	7
|10|	정보시스템	|7
|11	|운영	|7
|12	|재공고|	7
|13	|차세대	|5
|14	|차	|5
|15	|홈페이지|	5
|16	|년도	|5
|17	|구축사업|	4
|18	|수립	|4
|19	|통합	|4
|20	|전산시스템|	4
|21	|스마트	|4
|22	|관리시스템|	3
|23	|입찰공고|	3
|24	|통합정보시스템	|3
|25	|종합정보시스템	|3
|26	|단계	|3
|27	|연계	|3
|28	|개선|	3|
|29	|ISP|	3|

## 3.7.결측치 비율

In [ ]:
metadata_cols = df.columns.tolist()

meta_quality = pd.DataFrame({
    "결측치수": df[metadata_cols].isnull().sum(),
    "결측비율": round(
        df[metadata_cols].isnull().mean()*100,
        2
    )
})

display(meta_quality)

전처리 단계에서 <unknown> 태그로 채우고 사업 금액은 계산시 에러가 나서 0으로 채울예정

결과
||결측치수|	결측비율|
|---|---|---|
|공고 번호	|18|	18.0
|공고 차수	|18	|18.0
|사업명|	0	|0.0
|사업 금액	|1|	1.0
|발주 기관	|0|	0.0
|공개 일자	|0|	0.0
|입찰 참여 시작일|	26|	26.0
|입찰 참여 마감일	|8	|8.0
|사업 요약|	0|	0.0
|파일형식|	0	|0.0
|파일명	|0	|0.0
|텍스트	|0	|0.0
|문서길이	|0	|0.0
|목차존재|	0	|0.0

## 3.8.Chunking 규모 예측

Vector DB 크기 예측
Chunk Size 선정

In [ ]:
sizes = [500, 1000, 1500, 2000]

for size in sizes:

    chunk_count = (
        df["문서길이"] // size + 1
    ).sum()

    print(
        f"Chunk Size {size:>4} → "
        f"{int(chunk_count):>6} chunks"
    )

**결과**

Chunk Size  500 →    822 chunks

Chunk Size 1000 →    439 chunks

Chunk Size 1500 →    311 chunks

Chunk Size 2000 →    250 chunks

## 3.9.재공고 현황 확인

In [ ]:
df["재공고여부"] = df["공고 차수"].apply(
    lambda x: "재공고" if pd.notna(x) and x > 0 else "최초공고"
)

re_notice = df["재공고여부"].value_counts()

display(re_notice)

**결과**

||count|
|---|---|
|최초공고	|94|
|재공고	|6|


In [ ]:
# 재공고 목록 확인
df[df["공고 차수"] > 0][
    ["공고 번호", "공고 차수", "파일명"]
].sort_values("공고 차수", ascending=False)

**결과**

||공고 번호|	공고 차수|	파일명|
|---|---|---|---|
|61	|20241211469	|2.0|	케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp
|83|	20240311752|	2.0	|사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp
|25|	R25BK00601569	|1.0	|한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp
|19|	20240903676	|1.0	|고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp
|52|	20240531013	|1.0	|대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp
|37|	20240345257	|1.0	|전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp


# 3.EDA(원본 문서)

## 3.1.PDF / HWP 개수 확인

In [ ]:
from pathlib import Path

files_dir = Path(f"{BASE_DIR}/files")

pdf_count = 0
hwp_count = 0
hwpx_count = 0

for file in files_dir.rglob("*"):

    ext = file.suffix.lower()

    if ext == ".pdf":
        pdf_count += 1

    elif ext == ".hwp":
        hwp_count += 1

    elif ext == ".hwpx":
        hwpx_count += 1

print("PDF :", pdf_count)
print("HWP :", hwp_count)
print("HWPX:", hwpx_count)
print("전체:", pdf_count+hwp_count+hwpx_count)

**결과**

PDF : 4

HWP : 96

HWPX: 0

전체: 100